*0.1 Python for GenAI*

# Protocols

**The situation.** Your support product calls `openai_client.chat.completions.create(...)` directly in forty files. Then two things happen in one week: the company signs a contract with a second provider and wants 20% of traffic routed there, and the test suite is banned from calling the real API because it cost $300 last month. Forty files to edit, and no way to run a test without the network.

**The fix: depend on what you need, not on who provides it.** A *Protocol* describes a capability as a method signature — "something with `complete(prompt) -> str`". Your code is written against that. One small adapter per provider supplies it. A fake one supplies it in tests.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The capability, and two things that provide it.** Notice neither class mentions `ChatModel`. Having the method is enough — that is what "Protocol" means.

In [2]:
from typing import Protocol

from openai import OpenAI


class ChatModel(Protocol):
    def complete(self, prompt: str) -> str: ...


class OpenAIChatModel:
    def __init__(self) -> None:
        self.client = OpenAI(timeout=30)

    def complete(self, prompt: str) -> str:
        reply = self.client.chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0
        )
        return reply.choices[0].message.content or ""


class FakeChatModel:
    """For tests: returns a canned answer and remembers what it was asked."""

    def __init__(self, canned_answer: str) -> None:
        self.canned_answer = canned_answer
        self.prompts_seen = []

    def complete(self, prompt: str) -> str:
        self.prompts_seen.append(prompt)
        return self.canned_answer


print("two providers of the same capability:", OpenAIChatModel.__name__, FakeChatModel.__name__)

two providers of the same capability: OpenAIChatModel FakeChatModel


**The business code.** It takes a `ChatModel` and never imports a vendor library. This is the function the forty files should have been calling.

In [3]:
def summarise_ticket(model: ChatModel, ticket_text: str) -> str:
    return model.complete(f"Summarise this support ticket in one sentence: {ticket_text}")


ticket = "Customer says the password reset email never arrives; they checked spam; using Gmail."

print("production:", summarise_ticket(OpenAIChatModel(), ticket))

fake = FakeChatModel("Customer not receiving password reset emails on Gmail.")
print("in a test:  ", summarise_ticket(fake, ticket))
print("the test can check the prompt:", fake.prompts_seen[0][:45], "...")
assert fake.prompts_seen[0].startswith("Summarise this support ticket")

production: The customer is not receiving the password reset email in their Gmail account, even after checking their spam folder.
in a test:   Customer not receiving password reset emails on Gmail.
the test can check the prompt: Summarise this support ticket in one sentence ...


**Reading the output.** The same `summarise_ticket` produced a real summary through OpenAI and a canned one through the fake — no network, no cost — and the fake recorded the exact prompt so a test can assert on it. Adding the second provider is one new adapter class; the forty files do not change.

```
summarise_ticket(model, text)  ──▶  ChatModel.complete(prompt)
                                        ├── OpenAIChatModel   (production)
                                        ├── SecondProvider    (the new contract)
                                        └── FakeChatModel     (tests)
```

**The rule to remember.** Business code depends on a Protocol. Adapters depend on vendors. Tests use fakes.

| Use it when | Don't when | Instead use |
|---|---|---|
| a second provider, or a test without the network, exists or is coming | there will only ever be one implementation | call the vendor directly — an abstraction with one user is noise |

**Watch out**
- Keep each Protocol tiny: one capability. A 15-method interface is the vendor SDK in disguise.
- Run the same tests against the real adapter and the fake, or the fake drifts from reality.
- Do not put vendor-shaped arguments (OpenAI message dicts) in the Protocol — then nothing was decoupled.